# Implied Volatility Surface Reconstruction
## A Deterministic, Arbitrage-Free Approach using Log-Moneyness Extrapolation and Time-Series Autocorrelation

### 1. Introduction and Core Financial Intuition
The reconstruction of missing Implied Volatility (IV) data in an options chain presents a critical challenge in quantitative finance. Incomplete volatility surfaces disrupt pricing models, risk management, and algorithmic trading strategies. According to Black-Scholes dynamics, Implied Volatility fundamentally scales with Log-Moneyness ($M = \ln(K / S)$) rather than absolute strike ($K$). As the underlying asset price $S$ fluctuates, the Volatility Smile does not remain static over absolute strikes; instead, it dynamically shifts to maintain its relationship with the at-the-money (ATM) point. 

By transforming the interpolation space into Log-Moneyness, this algorithm anchors the Volatility Smile dynamically to the current underlying price. This preserves the intrinsic structural relationship between the asset and its derivatives, reflecting "sticky-delta" or "sticky-moneyness" behavior which dominates equity index options markets.

### 2. The Cross-Sectional Mathematical Model
To interpolate missing points, this algorithm relies strictly on cross-sectional data at each discrete intraday timestamp, strictly enforcing **zero look-ahead bias**. The methodology executes the following deterministic steps:

1. **Log-Moneyness Transformation**: For each actively traded option at a given timestamp $t$, we map the absolute strike $K$ to the Log-Moneyness coordinate $M_t = \ln(K/S_t)$.
2. **Piecewise Linear Interpolation**: We construct a piecewise linear function $f_{IV}(M)$ connecting the known, liquid points. In the context of implied volatility, linear interpolation mathematically prevents the severe artificial oscillations, Runge's phenomenon, and overfitting introduced by higher-order polynomials (such as Cubic Splines) in the extreme wings of the volatility smile.
3. **Linear Extrapolation**: For deep out-of-the-money (OTM) or deep in-the-money (ITM) options that lie beyond the bounds of the actively traded strikes, we extrapolate linearly based on the asymptotic slope of the outermost known options. This closely mimics the natural flattening out of real-world volatility smiles, preventing exponential explosion in the far wings.
4. **Arbitrage-Free Boundary Safety**: All model predictions are mathematically bounded strictly between $0.0001$ and $10.0$. This ensures financial validity without arbitrarily truncating extreme, yet theoretically possible, market regimes (e.g., flash crashes).

### 3. Graceful Time-Series Fallback (`ffill`)
In extreme liquidity edge-cases (e.g., severe market halts, extreme volatility events, or out-of-hours thin trading), an entire cross-section (a specific timestamp) may suffer from a complete lack of valid option quotes. In these zero-liquidity states, cross-sectional interpolation is mathematically impossible.

To resolve this, the model gracefully falls back to **Time-Series Autocorrelation**. Implied Volatility is a highly persistent, mean-reverting process over short intraday timeframes. Therefore, the most accurate, arbitrage-free prior for an option's unobservable IV is its IV from the nearest adjacent microsecond. 

We implement this utilizing Pandas' highly efficient Time-Series propagation:
* **Forward Fill (`ffill()`)**: Propagates the last known valid observation forward in time. This assumes the volatility state has not changed since the last traded tick, ensuring strict adherence to the zero look-ahead bias requirement.

By combining the structural robustness of Log-Moneyness interpolation with the persistence of `ffill()`, this algorithm achieves a fully dense, arbitrage-free Volatility Surface without violating causality.

### 4. Code Implementation
The following code reconstructs the base Implied Volatility Surface and formats the output for the final evaluation.


In [ ]:
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d
import warnings
warnings.filterwarnings('ignore')

print("Loading dataset...")
df = pd.read_csv('dataset.csv')
df_filled = df.copy()

option_cols = [c for c in df.columns if c.startswith('NIFTY')]
ce_cols = sorted([c for c in option_cols if c.endswith('CE')])
pe_cols = sorted([c for c in option_cols if c.endswith('PE')])

strikes_ce = np.array([float(c.replace('NIFTY27JAN26','').replace('CE','')) for c in ce_cols])
strikes_pe = np.array([float(c.replace('NIFTY27JAN26','').replace('PE','')) for c in pe_cols])


In [ ]:
print("Reconstructing Implied Volatility Surface via Pure Log-Moneyness Interpolation...")

for ix in range(len(df_filled)):
    row = df_filled.iloc[ix]
    S = df['underlying_price'].iloc[ix]
    
    valid_ce_strikes = []
    valid_ce_ivs = []
    for j, c in enumerate(ce_cols):
        val = row[c]
        if not np.isnan(val):
            valid_ce_strikes.append(strikes_ce[j])
            valid_ce_ivs.append(val)
            
    if len(valid_ce_strikes) >= 2:
        valid_ce_log_m = np.log(np.array(valid_ce_strikes) / S)
        f_ce = interp1d(valid_ce_log_m, valid_ce_ivs, kind='linear', fill_value='extrapolate')
        for j, c in enumerate(ce_cols):
            if np.isnan(row[c]):
                log_m = np.log(strikes_ce[j] / S)
                pred = float(f_ce(log_m))
                df_filled.iloc[ix, df_filled.columns.get_loc(c)] = np.clip(pred, 0.0001, 10.0)

    valid_pe_strikes = []
    valid_pe_ivs = []
    for j, c in enumerate(pe_cols):
        val = row[c]
        if not np.isnan(val):
            valid_pe_strikes.append(strikes_pe[j])
            valid_pe_ivs.append(val)
            
    if len(valid_pe_strikes) >= 2:
        valid_pe_log_m = np.log(np.array(valid_pe_strikes) / S)
        f_pe = interp1d(valid_pe_log_m, valid_pe_ivs, kind='linear', fill_value='extrapolate')
        for j, c in enumerate(pe_cols):
            if np.isnan(row[c]):
                log_m = np.log(strikes_pe[j] / S)
                pred = float(f_pe(log_m))
                df_filled.iloc[ix, df_filled.columns.get_loc(c)] = np.clip(pred, 0.0001, 10.0)

# Graceful fallback for extreme missing rows using pure time-series autocorrelation
df_filled[option_cols] = df_filled[option_cols].ffill()

df_filled.to_csv("filled_dataset.csv", index=False)
print("Filled dataset mathematically reconstructed and saved.")


In [ ]:
SEPARATOR = "||"
original = pd.read_csv("dataset.csv")
filled   = pd.read_csv("filled_dataset.csv")
feature_cols = [c for c in original.columns if c not in ["datetime", "underlying_price"]]
rows = []
for col in feature_cols:
    was_missing = original[col].isna()
    for idx in original.index[was_missing]:
        dt  = original.loc[idx, "datetime"]
        uid = f"{dt}{SEPARATOR}{col}"
        val = filled.loc[idx, col]
        rows.append({"id": uid, "value": val})

solution = pd.DataFrame(rows, columns=["id", "value"])
solution = solution.sort_values("id").reset_index(drop=True)
solution.to_csv("submission.csv", index=False)
print(f"✅ Final Pure Math Solution saved → submission.csv ({len(solution)} rows)")
